# Module 9: File Handling

**Utrains Python Fundamentals** &middot; lab notebook

*Read and write files, work safely with with, and handle JSON, including real API responses.*

## By the end of this notebook you can

- Pick the right file mode for what you are about to do
- Read a whole file, one line, or every line as a list
- Prefer with open(...) so files always get closed
- Turn Python data into JSON and back, as a string and as a file
- Locate a config file on disk with pathlib

## How to work through it

It follows the Module 9 slide deck, slide by slide.

The headings below are the slide numbers from the deck. The explanation for
each one is on the slide and in the [README](../README.md); this notebook is
where you run the code.

Run every cell in order with **Shift + Enter**.

Two cells are marked **Your turn**. They contain `____` where a piece of the
syntax is missing, so they fail if you run them as they are. That is
deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**: a short task with no code written for you.

**Assumed knowledge.** Modules 1 to 8. `import` appears throughout because `os`, `json` and `pathlib` need it; the deck flags this too, and Module 10 explains `import` properly.

### Before you start: the scratch folder

Everything in this notebook writes into a `scratch/` folder next to it, so
nothing else on your machine is touched. `scratch/` is in `.gitignore`, so none
of it will end up in a commit. Run this cell first.

In [ ]:
from pathlib import Path

WORK = Path("scratch")
WORK.mkdir(exist_ok=True)

print("writing files into:", WORK.resolve())

## Slide 2 &middot; What Is File Handling?

In [ ]:
# write something to disk
file = open(WORK / "notes.txt", "w")
file.write("Meeting at 3pm")
file.close()

# read it back later, even after the program restarts
file = open(WORK / "notes.txt", "r")
print(file.read())
file.close()

## Slide 4 &middot; Opening, Writing, and Reading

In [ ]:
file = open(WORK / "sample.txt", "w")
file.write("Hello, this is a test file.")
file.close()

file = open(WORK / "sample.txt", "r")
content = file.read()
print(content)
file.close()

## Slide 5 &middot; Reading Line by Line

In [ ]:
file = open(WORK / "sample.txt", "r")
line = file.readline()          # reads a single line
file.close()
print("readline :", repr(line))

file = open(WORK / "sample.txt", "r")
lines = file.readlines()        # reads all lines into a list
file.close()
print("readlines:", lines)

## Slide 6 &middot; The with Statement

In [ ]:
with open(WORK / "sample.txt", "r") as file:
    content = file.read()
    print(content)

# the file is already closed here, no need to call close()
print("closed?", file.closed)

## Slide 7 &middot; Writing Multiple Lines and Appending

In [ ]:
lines = ["Line 1\n", "Line 2\n", "Line 3\n"]

with open(WORK / "output.txt", "w") as file:
    file.writelines(lines)

with open(WORK / "output.txt", "a") as file:
    file.write("This line will be appended.\n")

with open(WORK / "output.txt", "r") as file:
    print(file.read())

---

### Your turn 1

Append an incident summary to a running log, then read the whole log back. Pick the mode that adds to the end rather than wiping the file.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
log_path = WORK / "incidents.log"

# TODO: choose the mode that ADDS to the end rather than overwriting.
with open(log_path, ____) as f:
    f.write("INC-4412 database connection failures resolved\n")

# TODO: choose the mode that reads.
with open(log_path, ____) as f:
    print(f.read())

Run that cell two or three times. The log grows each time, which is exactly what append mode is for. Change the mode to `"w"`, run it again, and watch the earlier lines disappear.

## Slide 8 &middot; Working with Binary Files

In [ ]:
# Make a small binary file so there is something real to copy.
with open(WORK / "image.jpg", "wb") as f:
    f.write(bytes(range(40)))

with open(WORK / "image.jpg", "rb") as file:
    data = file.read()
    print("Binary content:", data[:20])

with open(WORK / "copy.jpg", "wb") as new_file:
    new_file.write(data)

print("copy written, same size:", (WORK / "copy.jpg").stat().st_size == len(data))

## Slide 9 &middot; Checking and Deleting Files

In [ ]:
import os

target = WORK / "sample.txt"

if os.path.exists(target):
    print("File exists")
else:
    print("File not found")

os.remove(target)
print("after remove, exists?", os.path.exists(target))

## Slide 10 &middot; Working with JSON

In [ ]:
import json

person = {"name": "Alice", "age": 25}

as_text = json.dumps(person)
print(as_text)
print(type(as_text))

back_to_dict = json.loads(as_text)
print(back_to_dict["name"])
print(type(back_to_dict))

## Slide 11 &middot; Parsing a Real JSON Response

In [ ]:
raw_response = '''
{
  "model": "claude-sonnet-4-6",
  "content": [{"type": "text", "text": "The capital of France is Paris."}],
  "usage": {"input_tokens": 12, "output_tokens": 8}
}
'''

data = json.loads(raw_response)

print(data["content"][0]["text"])
print(data["usage"]["output_tokens"])

## Slide 12 &middot; Reading and Writing JSON Files

In [ ]:
conversation = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris."},
]

with open(WORK / "conversation.json", "w") as f:
    json.dump(conversation, f, indent=2)

with open(WORK / "conversation.json", "r") as f:
    loaded = json.load(f)

print(loaded[0]["content"])
print("turns saved:", len(loaded))

---

### Your turn 2

Save a dictionary of resource tags to a JSON file and read one value back. Watch carefully which of the four json functions each step needs.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
tags = {"env": "prod", "owner": "platform-team", "cost-center": "CC-1180"}

# TODO: write to a FILE, then read from a FILE. Mind the s.
with open(WORK / "tags.json", "w") as f:
    json.____(tags, f, indent=2)

with open(WORK / "tags.json", "r") as f:
    restored = json.____(f)

print(restored["owner"])

## Slide 13 &middot; Finding a Config File with pathlib

In [ ]:
from pathlib import Path

here = Path.cwd().resolve()
loaded_from = None

for candidate in [
    here / ".env",
    here.parent / ".env",
    here.parent.parent / ".env",
]:
    if candidate.is_file():
        loaded_from = candidate
        break

print("searched from:", here)
print("Would load .env from:", loaded_from)

### Reading a file that ships with this repo

`data/servers.txt` sits one folder up from this notebook, with one server name
per line. Reading a real file is the same code, just a different path.

In [ ]:
servers_file = Path("..") / "data" / "servers.txt"

with open(servers_file) as f:
    for line in f:
        print("-", line.strip())

---

# More use cases

The same ideas, applied to situations you will meet in real work. Run each one, then change a value and run it again.

## Use case 1 &middot; AI &middot; A run log, one JSON object per line

In [ ]:
import json

log_path = WORK / "runs.jsonl"

runs = [
    {"run": 1, "prompt": "summarise", "tokens": 128, "ok": True},
    {"run": 2, "prompt": "translate", "tokens": 96, "ok": True},
    {"run": 3, "prompt": "classify", "tokens": 64, "ok": False},
]

with open(log_path, "w") as f:
    for record in runs:
        f.write(json.dumps(record) + "\n")

total_tokens = 0
failures = 0

with open(log_path) as f:
    for line in f:
        record = json.loads(line)
        total_tokens += record["tokens"]
        if not record["ok"]:
            failures += 1

print("records    :", len(runs))
print("tokens used:", total_tokens)
print("failures   :", failures)

## Use case 2 &middot; AI &middot; Save a conversation and pick it up later

In [ ]:
import json

chat_path = WORK / "chat.json"

history = [
    {"role": "user", "content": "Hi, I am Serge."},
    {"role": "assistant", "content": "Hello Serge."},
]

with open(chat_path, "w") as f:
    json.dump(history, f, indent=2)

# ... the program stops and starts again here ...

with open(chat_path) as f:
    resumed = json.load(f)

resumed.append({"role": "user", "content": "What is my name?"})
resumed.append({"role": "assistant", "content": "You said you are Serge."})

with open(chat_path, "w") as f:
    json.dump(resumed, f, indent=2)

print("turns after resuming:", len(resumed))
print("last exchange:")
print("  ", resumed[-2]["content"])
print("  ", resumed[-1]["content"])

## Use case 3 &middot; AI &middot; Defaults, overridden by a config file

In [ ]:
import json

defaults = {"model": "gpt-4o-mini", "temperature": 0.7, "max_tokens": 512}

with open(WORK / "config.json", "w") as f:
    json.dump({"temperature": 0.2}, f)

with open(WORK / "config.json") as f:
    from_file = json.load(f)

settings = dict(defaults)
settings.update(from_file)

print("defaults :", defaults)
print("file says:", from_file)
print("in effect:", settings)

## Use case 4 &middot; Pull the errors out of a log file

In [ ]:
source = WORK / "app.log"
errors_only = WORK / "errors.log"

with open(source, "w") as f:
    f.writelines([
        "2024-08-19 INFO  service started\n",
        "2024-08-19 ERROR database connection refused\n",
        "2024-08-19 INFO  retrying\n",
        "2024-08-19 ERROR database connection refused\n",
        "2024-08-19 INFO  connected\n",
    ])

kept = 0

with open(source) as src, open(errors_only, "w") as dst:
    for line in src:
        if "ERROR" in line:
            dst.write(line)
            kept += 1

print("errors found:", kept)

with open(errors_only) as f:
    print(f.read())

---

## Lab: A server inventory round trip

Read `../data/servers.txt`, which holds one server name per line.

Turn it into a list of dictionaries, where each entry has a `name`, a `region`
taken from the part of the name after the first hyphen that follows the number,
and a `status` of `"unknown"`.

Save that list to `scratch/inventory.json` with an indent of 2, then read it
back from disk into a fresh variable, and print how many servers you recovered
and the name of the last one.

Finish by appending one audit line to `scratch/audit.log` recording how many
servers were processed. Run the whole lab twice: the audit log should have two
lines while the inventory JSON still holds the right count.

**Done when:**

- [ ] The text file is read with a with statement, not open/close
- [ ] Each line becomes a dictionary in a list
- [ ] json.dump writes the file and json.load reads it back
- [ ] The audit log uses append mode and grows on a second run

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

The four exercises from the module's practice slide are in the
[README](../README.md#practice-exercises) and repeated on the slide. There are
4 of them. Do them in a scratch cell here or in a `.py` file.

## Module complete

You can now read, write and manage files, and work with JSON data confidently.

*Utrains &middot; support@utrains.org &middot; https://utrains.org*